# Lab 17 — External Evaluation on DeepFakeDetection (DFD)

Deepfake Detection for Video KYC.

## 2. Experiment objective

Labs 14-16 trained and selected models entirely on the **core** FaceForensics++ split
(350/75/75 identity groups for train/validation/test). The **DeepFakeDetection (DFD)**
subset — 1,000 videos, all FAKE, from a separate actor/identity namespace — was
deliberately excluded from that core split from the very beginning
(`src/preprocessing/dataset_splitter.py`) and has never been seen by any model in any
capacity: not training, not validation, not core test evaluation, not checkpoint
selection.

Lab 17 uses DFD as a genuinely **external** evaluation set: a distribution the model has
never been optimized against, sourced from a different subset of the same underlying
FaceForensics++ collection. This measures whether the detector's performance holds up
outside the exact identities/manipulations it was tuned on, or whether it was overfit to
the core split's specific distribution.

## 3. Research question

**"How well does the trained deepfake detector generalize to the held-out
DeepFakeDetection dataset that was not used during training, validation, or core
testing?"**

## 4. Experimental design

- Models were trained and selected exclusively on the core FF++ identity-disjoint split
  (Lab 14 MobileNetV2, Lab 15 EfficientNet-B0, Lab 16 class-balanced EfficientNet-B0).
- DFD was **not** used during training, validation, or core-test model selection at any
  point in Labs 14-16.
- DFD contains **1,000 videos, all labeled FAKE** — there are no REAL examples in DFD.
- This notebook performs **external evaluation only**: existing checkpoints are loaded
  and run in inference mode (`model.eval()`, `torch.no_grad()`).
- **No retraining, no fine-tuning, no gradient updates, and no use of DFD for model
  selection** happen anywhere in this notebook.
- Because DFD is all-FAKE, standard two-class metrics (accuracy, ROC-AUC, REAL
  precision/recall) are either undefined or misleading here. The primary external metric
  is **FAKE recall / FAKE detection rate**, reported at both the image level and the
  video level.

## 5. Environment setup

Designed for a fresh Google Colab runtime with a T4 GPU. No Google Drive mounting is
required. This notebook downloads the **raw FaceForensics++ C23 video dataset**
(`xdxd003/ff-c23`) — the same source Lab 13 used to build the processed train/val/test
image dataset — because DFD videos were never processed into the
`aishwarya99990/faceforensics-kyc-processed` image dataset that Labs 14-16 consume. Only
the `DeepFakeDetection/` videos are extracted from the archive, not the full dataset.

In [1]:
!pip uninstall -y opencv-python opencv-contrib-python opencv-python-headless opencv-contrib-python-headless
!pip install -q opencv-python==4.10.0.84

Found existing installation: opencv-python 5.0.0.93
Uninstalling opencv-python-5.0.0.93:
  Successfully uninstalled opencv-python-5.0.0.93
Found existing installation: opencv-contrib-python 4.14.0.94
Uninstalling opencv-contrib-python-4.14.0.94:
  Successfully uninstalled opencv-contrib-python-4.14.0.94
Found existing installation: opencv-python-headless 5.0.0.93
Uninstalling opencv-python-headless-5.0.0.93:
  Successfully uninstalled opencv-python-headless-5.0.0.93
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.5/62.5 MB 11.6 MB/s eta 0:00:00


### Restart the Colab runtime now

The cell above uninstalled the Colab-default OpenCV build and installed
`opencv-python==4.10.0.84` (the version this project's face detector requires).
Because `cv2` has not been imported yet in this process, this usually works without a
restart - but to avoid any risk of a stale/mismatched `cv2` binary being cached in this
runtime, restart before verifying it:

**Runtime -> Restart session** (Colab menu), then continue running the notebook from
the next cell onward. Do **not** use "Restart and run all" - just restart the session,
then keep running cells top to bottom manually from here.


In [2]:
import cv2

print("OpenCV version:", cv2.__version__)
print("CascadeClassifier available:", hasattr(cv2, "CascadeClassifier"))

assert cv2.__version__ == "4.10.0"
assert hasattr(cv2, "CascadeClassifier")

print("OpenCV verification PASSED")

OpenCV version: 4.10.0
CascadeClassifier available: True
OpenCV verification PASSED


In [3]:
!pip install -q kaggle scikit-learn

In [4]:
import os
import sys
import json
import shutil
import zipfile
from pathlib import Path
from getpass import getpass

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

print("Running in Colab:", IN_COLAB)

Running in Colab: True


In [5]:
# Clone (or reuse) the project repository so we have access to
# src.preprocessing / src.dataset and the rest of the project code.
REPO_URL = "https://github.com/Aishwarya-93/Deepfake-Detection-KYC.git"
REPO_NAME = "Deepfake-Detection-KYC"

if IN_COLAB:
    os.chdir("/content")
    if not Path(REPO_NAME).exists():
        exit_code = os.system(f"git clone {REPO_URL}")
        if exit_code != 0:
            raise RuntimeError("git clone failed. Check REPO_URL / network access.")
    os.chdir(REPO_NAME)
else:
    print("Not running in Colab - assuming the notebook already sits inside the repo.")

print("Working directory:", Path.cwd())

Working directory: /content/Deepfake-Detection-KYC


In [6]:
!ls -lah data/splits/

total 324K
drwxr-xr-x 2 root root 4.0K Sep  8 10:03 .
drwxr-xr-x 7 root root 4.0K Sep  8 10:03 ..
-rw-r--r-- 1 root root    0 Sep  8 10:03 .gitkeep
-rw-r--r-- 1 root root  47K Sep  8 10:03 test.csv
-rw-r--r-- 1 root root 219K Sep  8 10:03 train.csv
-rw-r--r-- 1 root root  47K Sep  8 10:03 validation.csv


In [7]:
def find_project_root(start: Path) -> Path:
    """Walk upward from `start` until a directory containing both
    `src/` and `data/` is found. Falls back to `start` if not found.
    Avoids hard-coding any machine-specific path."""
    for directory in [start] + list(start.parents):
        if (directory / "src").is_dir() and (directory / "data").is_dir():
            return directory
    return start


PROJECT_ROOT = find_project_root(Path.cwd())

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Project root:", PROJECT_ROOT)

for required in [
    "src/preprocessing/frame_extractor.py",
    "src/preprocessing/face_detector.py",
    "src/preprocessing/image_preprocessing.py",
    "src/dataset/dataloader.py",
]:
    assert (PROJECT_ROOT / required).exists(), f"Missing expected project file: {required}"

print("Project structure verified (preprocessing/model files present).")

Project root: /content/Deepfake-Detection-KYC
Project structure verified (preprocessing/model files present).


## 6. Imports

Reuses the project's existing preprocessing modules (`frame_extractor`, `face_detector`,
`image_preprocessing`) and dataset class (`FaceDataset`) rather than inventing a second,
possibly-contradictory pipeline.

In [8]:
from collections import Counter, defaultdict
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import cv2
import torch
import torchvision
from torch import nn
from torch.utils.data import DataLoader
from torchvision import transforms
from torchvision.models import (
    efficientnet_b0, EfficientNet_B0_Weights,
    mobilenet_v2, MobileNet_V2_Weights,
)
import matplotlib.pyplot as plt

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    confusion_matrix,
    roc_auc_score,
)

from src.preprocessing import face_detector as face_detector_module
from src.preprocessing import image_preprocessing
from src.dataset.dataloader import FaceDataset

print("PyTorch version:     ", torch.__version__)
print("Torchvision version: ", torchvision.__version__)

if torch.cuda.is_available():
    print("CUDA available: Yes")
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("CUDA available: No (Runtime > Change runtime type > T4 GPU is recommended)")

PyTorch version:      2.11.0+cu128
Torchvision version:  0.26.0+cu128
CUDA available: Yes
GPU: Tesla T4


## 7. Reproducibility / random seeds

`SEED = 42` matches the convention used throughout this project (`DEFAULT_RANDOM_STATE`
in `src/preprocessing/dataset_splitter.py`, and the seed used in Lab 16). This notebook
performs no training, so the seed mainly affects nothing stochastic in inference itself —
it is set anyway for consistency with the rest of the project and in case any future cell
introduces randomness (e.g. sampling a subset for a plot).

In [9]:
import random

SEED = 42

def set_seed(seed: int = SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(SEED)
print("Random seed set to:", SEED)

Random seed set to: 42


## 8. Kaggle authentication

Same `KAGGLE_API_TOKEN` Colab secret used in Labs 14-16.

In [11]:
def load_kaggle_credentials():
    """Authenticate with Kaggle using the existing KAGGLE_API_TOKEN Colab
    secret. Falls back to an interactive prompt only if that secret isn't
    available."""

    token = None
    username = None

    if IN_COLAB:
        try:
            from google.colab import userdata
            token = userdata.get("KAGGLE_API_TOKEN")
        except Exception:
            token = None

    if token:
        os.environ["KAGGLE_API_TOKEN"] = token
        os.environ["KAGGLE_KEY"] = token

        if IN_COLAB:
            try:
                username = userdata.get("KAGGLE_USERNAME")
            except Exception:
                username = None
        if username:
            os.environ["KAGGLE_USERNAME"] = username

        print("Kaggle token loaded:", bool(token))
    else:
        print("KAGGLE_API_TOKEN secret not found.")
        print("Falling back to interactive input.")
        username = input("Kaggle username: ").strip()
        key = getpass("Kaggle API key: ").strip()

        kaggle_dir = Path.home() / ".kaggle"
        kaggle_dir.mkdir(parents=True, exist_ok=True)
        kaggle_json_path = kaggle_dir / "kaggle.json"
        kaggle_json_path.write_text(json.dumps({"username": username, "key": key}))
        kaggle_json_path.chmod(0o600)

        os.environ["KAGGLE_USERNAME"] = username
        os.environ["KAGGLE_KEY"] = key

        print("Kaggle credentials configured:", bool(username and key))


load_kaggle_credentials()

Kaggle token loaded: True


## 9. Dataset download

Downloads the **raw** `xdxd003/ff-c23` dataset (the same one Lab 13 used) — this is a
large archive (~16.7 GB, as already observed in Lab 13's own run), so this cell skips the
download entirely if a `DeepFakeDetection` folder with the expected video count is
already found under `data/raw/` in this runtime.

In [12]:
RAW_DATASET_SLUG = "xdxd003/ff-c23"
RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"
RAW_DATA_DIR.mkdir(parents=True, exist_ok=True)

EXPECTED_DFD_VIDEO_COUNT = 1000


def find_dfd_video_dir(root: Path):
    """Locate an extracted 'DeepFakeDetection' folder anywhere under root,
    without assuming a specific parent-folder name in the archive layout."""
    for candidate in root.rglob("DeepFakeDetection"):
        if candidate.is_dir():
            return candidate
    return None


existing_dfd_dir = find_dfd_video_dir(RAW_DATA_DIR)
existing_mp4_count = (
    len(list(existing_dfd_dir.glob("*.mp4"))) if existing_dfd_dir is not None else 0
)

if existing_dfd_dir is not None and existing_mp4_count >= EXPECTED_DFD_VIDEO_COUNT:
    print(f"DeepFakeDetection videos already present at: {existing_dfd_dir}")
    print(f"Found {existing_mp4_count} mp4 files. Skipping download.")
    ZIP_PATH = None
else:
    print(f"DeepFakeDetection videos not found (or incomplete: {existing_mp4_count} found).")
    print("Downloading xdxd003/ff-c23 from Kaggle (large: ~16.7 GB)...")

    ZIP_PATH = PROJECT_ROOT / "ff-c23.zip"

    exit_code = os.system(
        f'kaggle datasets download -d {RAW_DATASET_SLUG} -p "{PROJECT_ROOT}"'
    )
    if exit_code != 0:
        raise RuntimeError(
            "kaggle datasets download failed. Check your Kaggle credentials "
            "(Section 8) and that you have accepted the dataset's terms on Kaggle."
        )
    if not ZIP_PATH.exists():
        raise RuntimeError(f"Expected archive not found after download: {ZIP_PATH}")

    print("Download complete:", ZIP_PATH)

DeepFakeDetection videos not found (or incomplete: 0 found).
Download complete: /content/Deepfake-Detection-KYC/ff-c23.zip


## 10. Dataset extraction

Extracts **only** the `DeepFakeDetection/` entries from the archive (not the full
multi-manipulation-method dataset), to avoid unnecessarily consuming Colab disk/time on
videos this notebook does not need.

In [13]:
if existing_dfd_dir is not None and existing_mp4_count >= EXPECTED_DFD_VIDEO_COUNT:
    DFD_VIDEO_DIR = existing_dfd_dir
    print("Reusing already-extracted videos at:", DFD_VIDEO_DIR)
else:
    print("Scanning archive for DeepFakeDetection entries...")
    with zipfile.ZipFile(ZIP_PATH) as zf:
        names = zf.namelist()
        dfd_members = [
            n for n in names
            if "/DeepFakeDetection/" in n or n.startswith("DeepFakeDetection/")
        ]

        if not dfd_members:
            raise RuntimeError(
                "Could not find any 'DeepFakeDetection' entries inside the archive. "
                f"First archive entries were: {names[:10]}. "
                "Inspect the archive layout before proceeding."
            )

        print(f"Found {len(dfd_members)} DeepFakeDetection archive entries. Extracting...")
        for member in dfd_members:
            zf.extract(member, path=RAW_DATA_DIR)

    DFD_VIDEO_DIR = find_dfd_video_dir(RAW_DATA_DIR)
    if DFD_VIDEO_DIR is None:
        raise RuntimeError(
            f"Extraction completed but no 'DeepFakeDetection' folder was found under {RAW_DATA_DIR}."
        )
    print("Extracted to:", DFD_VIDEO_DIR)

extracted_mp4_count = len(list(DFD_VIDEO_DIR.glob("*.mp4")))
print("mp4 files found:", extracted_mp4_count)

Scanning archive for DeepFakeDetection entries...
Found 1000 DeepFakeDetection archive entries. Extracting...
Extracted to: /content/Deepfake-Detection-KYC/data/raw/FaceForensics++_C23/DeepFakeDetection
mp4 files found: 1000


## 11. Dataset path verification

In [14]:
print("DFD_VIDEO_DIR:", DFD_VIDEO_DIR)
print("Exists:", DFD_VIDEO_DIR.exists())
print("mp4 count:", extracted_mp4_count)

if not DFD_VIDEO_DIR.exists():
    raise FileNotFoundError(f"DFD_VIDEO_DIR does not exist: {DFD_VIDEO_DIR}")

if extracted_mp4_count < EXPECTED_DFD_VIDEO_COUNT:
    raise RuntimeError(
        f"Only {extracted_mp4_count} DFD videos found, expected at least "
        f"{EXPECTED_DFD_VIDEO_COUNT}. Extraction may be incomplete."
    )

# DATASET_ROOT is the folder that CSV 'File Path' values are relative to
# (it directly contains DeepFakeDetection/, matching the convention used in Lab 13).
DATASET_ROOT = DFD_VIDEO_DIR.parent
print("DATASET_ROOT:", DATASET_ROOT)

DFD_VIDEO_DIR: /content/Deepfake-Detection-KYC/data/raw/FaceForensics++_C23/DeepFakeDetection
Exists: True
mp4 count: 1000
DATASET_ROOT: /content/Deepfake-Detection-KYC/data/raw/FaceForensics++_C23


## 12. Load the held-out DFD manifest

If the existing DFD split file is present, it is **read only** — never modified. If it is absent on the current branch, the notebook deterministically builds a 1,000-video external manifest from the extracted DFD videos and saves it only under `data/processed/external_dfd/`.

The existing, already-established
DFD split (built by `src/preprocessing/dataset_splitter.py`). Columns are inspected
rather than assumed, since the CSV's actual schema
(`File Path, Label, Frame Count, Width, Height, Identity Namespace, Identity Ids,
Identity Group`) was confirmed by inspecting the repository before writing this
notebook.

In [15]:
DFD_SPLIT_CSV = PROJECT_ROOT / "data" / "splits" / "deepfakedetection_test.csv"

if DFD_SPLIT_CSV.exists():
    # Preferred path: use the established split as a read-only artifact.
    dfd_df = pd.read_csv(DFD_SPLIT_CSV)
    print("Using established DFD split:", DFD_SPLIT_CSV)
else:
    # Fallback for a clean/current branch where the split CSV is not tracked.
    # Deterministically enumerate the extracted DFD videos; do not modify data/splits/.
    print("Established DFD split CSV not found. Building a deterministic external manifest from extracted DFD videos.")
    dfd_paths = sorted(DFD_VIDEO_DIR.rglob("*.mp4"))
    if len(dfd_paths) != EXPECTED_DFD_VIDEO_COUNT:
        raise RuntimeError(
            f"Expected exactly {EXPECTED_DFD_VIDEO_COUNT} extracted DFD videos, found {len(dfd_paths)}."
        )
    dfd_df = pd.DataFrame({
        "File Path": [str(p.relative_to(DATASET_ROOT)).replace("\\\\", "/") for p in dfd_paths],
        "Label": ["FAKE"] * len(dfd_paths),
    })
    fallback_path = PROJECT_ROOT / "data" / "processed" / "external_dfd" / "dfd_external_manifest.csv"
    fallback_path.parent.mkdir(parents=True, exist_ok=True)
    dfd_df.to_csv(fallback_path, index=False)
    DFD_SPLIT_CSV = fallback_path
    print("Fallback manifest saved to:", DFD_SPLIT_CSV)

print("Columns:", dfd_df.columns.tolist())
print("Rows:", len(dfd_df))
print()
print(dfd_df.head())
print()
print("Label distribution:")
print(dfd_df["Label"].value_counts())

REQUIRED_COLUMNS = ["File Path", "Label"]
for col in REQUIRED_COLUMNS:
    if col not in dfd_df.columns:
        raise RuntimeError(
            f"Expected column '{col}' not found in {DFD_SPLIT_CSV}. "
            f"Actual columns: {dfd_df.columns.tolist()}. Stopping rather than guessing."
        )


Established DFD split CSV not found. Building a deterministic external manifest from extracted DFD videos.
Fallback manifest saved to: /content/Deepfake-Detection-KYC/data/processed/external_dfd/dfd_external_manifest.csv
Columns: ['File Path', 'Label']
Rows: 1000

                                           File Path Label
0  DeepFakeDetection/01_02__meeting_serious__YVGY...  FAKE
1  DeepFakeDetection/01_02__outside_talking_still...  FAKE
2  DeepFakeDetection/01_02__talking_against_wall_...  FAKE
3  DeepFakeDetection/01_02__walk_down_hall_angry_...  FAKE
4  DeepFakeDetection/01_02__walking_down_indoor_h...  FAKE

Label distribution:
Label
FAKE    1000
Name: count, dtype: int64


## 13. DFD dataset statistics

Verifies the CSV matches the expectations this notebook is designed around. If it does
not, this cell **stops with a clear error** instead of silently proceeding on
unexpected data.

In [16]:
if len(dfd_df) != EXPECTED_DFD_VIDEO_COUNT:
    raise RuntimeError(
        f"Expected exactly {EXPECTED_DFD_VIDEO_COUNT} rows in {DFD_SPLIT_CSV}, "
        f"found {len(dfd_df)}. Stopping - the DFD split file appears to have changed."
    )

unique_labels = set(dfd_df["Label"].astype(str).str.strip().str.upper().unique())
if unique_labels != {"FAKE"}:
    raise RuntimeError(
        f"Expected every DFD label to be exactly 'FAKE', found label set: {unique_labels}. "
        "Stopping - do not silently coerce unexpected labels."
    )

print(f"Row count OK: {len(dfd_df)} == {EXPECTED_DFD_VIDEO_COUNT}")
print(f"Label check OK: all {len(dfd_df)} rows are FAKE, 0 REAL rows.")

if "Identity Namespace" in dfd_df.columns:
    print("\nIdentity Namespace values:", dfd_df["Identity Namespace"].unique().tolist())
if "Identity Group" in dfd_df.columns:
    non_null_groups = dfd_df["Identity Group"].dropna()
    print(
        "Identity Group is populated for", len(non_null_groups), "/", len(dfd_df),
        "rows (expected 0 - DFD was excluded from the 350/75/75 core identity-group split)."
    )

Row count OK: 1000 == 1000
Label check OK: all 1000 rows are FAKE, 0 REAL rows.


## 14. Frame / face processing

Reuses the project's existing preprocessing building blocks rather than a second,
parallel pipeline:

- `src/preprocessing/face_detector.py` — `load_face_detector`, `detect_faces`,
  `crop_faces` (OpenCV Haar Cascade), used exactly as in Lab 13.
- `src/preprocessing/image_preprocessing.py` — `resize_image` for the 224x224 face crop.
- Frame sampling (`frame_interval=10`, `max_frames_per_video=5`) and the collision-safe
  filename scheme (`build_source_id` — a sanitized, full-relative-path-based identifier)
  are reused **verbatim from the fix already made in Lab 13**
  (`notebooks/Lab_13_Dataset_Generation.ipynb`), so DFD filenames can never collide with
  each other and every output file is traceable back to its exact source video.

Output goes to a **new, separate** directory:

`data/processed/external_dfd/fake/`

`data/processed/train/`, `data/processed/val/`, and `data/processed/test/` are never
opened for writing anywhere in this notebook. Since DFD is 100% FAKE, only a `fake/`
subfolder is created — `FaceDataset` (Section 16) already handles a dataset with no
`real/` subfolder by yielding zero REAL samples, so no new Dataset class is needed.

## 15. External dataset processing

Processes every video in the DFD split into face images, building:
- `dfd_manifest.csv` — one row per **saved face image**: source video, `source_id`,
  frame index, face index, output filename. This is the traceable
  video -> frame -> face -> image mapping used for video-level aggregation in Section 20.
- `dfd_processing_log.csv` — one row per **video** (whether or not it produced any
  images), recording its status, so failures are reported rather than silently dropped.

Idempotent: if a manifest from a previous run in this runtime already accounts for all
1,000 videos, processing is skipped.

In [17]:
import re


def build_source_id(video_path: Path, dataset_root: Path) -> str:
    """Deterministic, globally-unique identifier for a source video, built
    from its path relative to DATASET_ROOT. Identical approach to the
    collision fix in Lab 13 (notebooks/Lab_13_Dataset_Generation.ipynb):
    using the full relative path (not just video_path.stem) keeps
    identifiers unique across folders that might reuse the same video stem.
    """
    relative_path = video_path.relative_to(dataset_root).with_suffix("")
    sanitized_parts = [
        re.sub(r"[^A-Za-z0-9_-]", "_", part)
        for part in relative_path.parts
    ]
    return "_".join(sanitized_parts)


EXTERNAL_DFD_ROOT = PROJECT_ROOT / "data" / "processed" / "external_dfd"
EXTERNAL_DFD_FAKE_DIR = EXTERNAL_DFD_ROOT / "fake"
EXTERNAL_DFD_FAKE_DIR.mkdir(parents=True, exist_ok=True)

MANIFEST_PATH = EXTERNAL_DFD_ROOT / "dfd_manifest.csv"
PROCESSING_LOG_PATH = EXTERNAL_DFD_ROOT / "dfd_processing_log.csv"

FRAME_INTERVAL = 10
MAX_FRAMES_PER_VIDEO = 5

print("Video source :", DATASET_ROOT)
print("Image output :", EXTERNAL_DFD_FAKE_DIR)
print("Frame interval:", FRAME_INTERVAL, "| Max frames/video:", MAX_FRAMES_PER_VIDEO)

Video source : /content/Deepfake-Detection-KYC/data/raw/FaceForensics++_C23
Image output : /content/Deepfake-Detection-KYC/data/processed/external_dfd/fake
Frame interval: 10 | Max frames/video: 5


In [18]:
def process_dfd_split(df, dataset_root, output_dir, frame_interval, max_frames_per_video):
    """Process every DFD video into face images, mirroring Lab 13's
    process_dataset_split (same source_id + collision-safe filename scheme),
    while recording per-video status and a full image-level manifest so
    nothing is silently dropped."""

    detector = face_detector_module.load_face_detector()

    manifest_rows = []
    processing_log = []
    written_sources = {}

    for _, row in df.iterrows():
        relative_path = Path(row["File Path"])
        label = str(row["Label"]).strip().lower()
        video_path = dataset_root / relative_path
        source_id = build_source_id(video_path, dataset_root)

        log_entry = {
            "file_path": str(relative_path),
            "source_id": source_id,
            "label": label,
            "status": None,
            "frames_read": 0,
            "faces_saved": 0,
        }

        if not video_path.exists():
            log_entry["status"] = "missing"
            processing_log.append(log_entry)
            continue

        cap = cv2.VideoCapture(str(video_path))
        if not cap.isOpened():
            log_entry["status"] = "unreadable"
            processing_log.append(log_entry)
            continue

        frame_index = 0
        frames_read = 0
        frames_used = 0
        faces_saved_for_video = 0

        while True:
            success, frame = cap.read()
            if not success:
                break
            frames_read += 1

            if frame_index % frame_interval == 0:
                faces = face_detector_module.detect_faces(frame, detector)
                cropped_faces = face_detector_module.crop_faces(frame, faces)

                for face_index, face in enumerate(cropped_faces):
                    processed_face = image_preprocessing.resize_image(face, (224, 224))

                    output_file = (
                        output_dir
                        / f"{source_id}_frame{frame_index}_face{face_index}.jpg"
                    )

                    if output_file.exists():
                        existing_source = written_sources.get(output_file)
                        if existing_source != str(video_path):
                            raise RuntimeError(
                                f"Filename collision detected: {output_file} already "
                                f"exists (written by {existing_source or 'an unknown prior run'}), "
                                f"but is now requested by {video_path}. Refusing to "
                                "silently overwrite."
                            )

                    cv2.imwrite(str(output_file), processed_face)
                    written_sources[output_file] = str(video_path)

                    manifest_rows.append({
                        "video_file_path": str(relative_path),
                        "source_id": source_id,
                        "label": label,
                        "frame_index": frame_index,
                        "face_index": face_index,
                        "output_filename": output_file.name,
                    })
                    faces_saved_for_video += 1

                frames_used += 1
                if frames_used >= max_frames_per_video:
                    break

            frame_index += 1

        cap.release()

        log_entry["frames_read"] = frames_read
        log_entry["faces_saved"] = faces_saved_for_video

        if frames_read == 0:
            log_entry["status"] = "no_frames_read"
        elif faces_saved_for_video == 0:
            log_entry["status"] = "no_faces_detected"
        else:
            log_entry["status"] = "ok"

        processing_log.append(log_entry)

        if len(processing_log) % 100 == 0:
            print(f"Processed {len(processing_log)}/{len(df)} DFD videos...")

    return pd.DataFrame(manifest_rows), pd.DataFrame(processing_log)


existing_manifest_ok = (
    MANIFEST_PATH.exists()
    and PROCESSING_LOG_PATH.exists()
    and len(pd.read_csv(PROCESSING_LOG_PATH)) == len(dfd_df)
)

if existing_manifest_ok:
    # Manifest/log row counts look right, but also confirm the actual face
    # images are still present in data/processed/external_dfd/fake/ before
    # trusting the cache - a manifest without matching images cannot be
    # reused (e.g. a fresh runtime with only the CSVs restored, but not the
    # JPEGs).
    candidate_manifest = pd.read_csv(MANIFEST_PATH)
    existing_image_count = len(list(EXTERNAL_DFD_FAKE_DIR.glob("*.jpg")))
    if existing_image_count != len(candidate_manifest):
        print(
            f"Manifest/log found, but {EXTERNAL_DFD_FAKE_DIR} has "
            f"{existing_image_count} image(s) vs {len(candidate_manifest)} manifest "
            "rows - treating the cache as incomplete and reprocessing."
        )
        existing_manifest_ok = False

if existing_manifest_ok:
    print(
        "Existing processed DFD images + manifest/log found for all", len(dfd_df),
        "videos at", EXTERNAL_DFD_FAKE_DIR, "- skipping reprocessing."
    )
    dfd_manifest = candidate_manifest
    dfd_processing_log = pd.read_csv(PROCESSING_LOG_PATH)
else:
    print("Processing DFD videos into face images (this can take a while)...")
    dfd_manifest, dfd_processing_log = process_dfd_split(
        dfd_df, DATASET_ROOT, EXTERNAL_DFD_FAKE_DIR, FRAME_INTERVAL, MAX_FRAMES_PER_VIDEO
    )
    dfd_manifest.to_csv(MANIFEST_PATH, index=False)
    dfd_processing_log.to_csv(PROCESSING_LOG_PATH, index=False)
    print("Saved manifest to:", MANIFEST_PATH)
    print("Saved processing log to:", PROCESSING_LOG_PATH)

Processing DFD videos into face images (this can take a while)...
Processed 100/1000 DFD videos...
Processed 200/1000 DFD videos...
Processed 300/1000 DFD videos...
Processed 400/1000 DFD videos...
Processed 500/1000 DFD videos...
Processed 600/1000 DFD videos...
Processed 700/1000 DFD videos...
Processed 800/1000 DFD videos...
Processed 900/1000 DFD videos...
Processed 1000/1000 DFD videos...
Saved manifest to: /content/Deepfake-Detection-KYC/data/processed/external_dfd/dfd_manifest.csv
Saved processing log to: /content/Deepfake-Detection-KYC/data/processed/external_dfd/dfd_processing_log.csv


In [19]:
status_counts = dfd_processing_log["status"].value_counts().to_dict()

print("===================================")
print("DFD PROCESSING REPORT")
print("===================================")
print("Videos in split          :", len(dfd_df))
print("Videos processed (any)   :", len(dfd_processing_log))
print("  ok (>=1 face saved)    :", status_counts.get("ok", 0))
print("  missing                :", status_counts.get("missing", 0))
print("  unreadable              :", status_counts.get("unreadable", 0))
print("  no_frames_read          :", status_counts.get("no_frames_read", 0))
print("  no_faces_detected       :", status_counts.get("no_faces_detected", 0))
print()
print("Total frames sampled     :", int(dfd_processing_log["frames_read"].sum()))
print("Total faces saved        :", int(dfd_processing_log["faces_saved"].sum()))
print("Final manifest rows      :", len(dfd_manifest))

failed_videos_log = dfd_processing_log[dfd_processing_log["status"] != "ok"]
if len(failed_videos_log) > 0:
    print(f"\n{len(failed_videos_log)} video(s) produced zero usable face images:")
    print(failed_videos_log[["file_path", "status", "frames_read", "faces_saved"]].to_string(index=False))
else:
    print("\nEvery DFD video produced at least one usable face image.")

DFD PROCESSING REPORT
Videos in split          : 1000
Videos processed (any)   : 1000
  ok (>=1 face saved)    : 994
  missing                : 0
  unreadable              : 0
  no_frames_read          : 0
  no_faces_detected       : 6

Total frames sampled     : 40863
Total faces saved        : 8446
Final manifest rows      : 8446

6 video(s) produced zero usable face images:
                                                 file_path            status  frames_read  faces_saved
DeepFakeDetection/01_11__secret_conversation__4OJNJLOO.mp4 no_faces_detected           41            0
DeepFakeDetection/02_01__secret_conversation__YVGY8LOK.mp4 no_faces_detected           41            0
DeepFakeDetection/02_07__secret_conversation__0IYV5DQ5.mp4 no_faces_detected           41            0
DeepFakeDetection/02_13__secret_conversation__CP5HFV3K.mp4 no_faces_detected           33            0
DeepFakeDetection/02_18__secret_conversation__B95S4G6F.mp4 no_faces_detected           41            0
  

### Corrupted/zero-byte image scan

Same integrity check used before building a `Dataset`/`DataLoader` in Labs 14-16, scoped
**only** to `external_dfd/` — `data/processed/train|val|test` are never touched.

In [20]:
from PIL import Image

def find_bad_images(folder: Path):
    bad = []
    for path in folder.glob("*.jpg"):
        try:
            if path.stat().st_size == 0:
                bad.append((path, "zero-byte file"))
                continue
            with Image.open(path) as img:
                img.verify()
            with Image.open(path) as img:
                img.convert("RGB").load()
        except Exception as e:
            bad.append((path, repr(e)))
    return bad


QUARANTINE_DIR = EXTERNAL_DFD_ROOT / "quarantine"
bad_images = find_bad_images(EXTERNAL_DFD_FAKE_DIR)
print("Corrupted/unreadable images found:", len(bad_images))

if bad_images:
    QUARANTINE_DIR.mkdir(parents=True, exist_ok=True)
    for path, reason in bad_images:
        dest = QUARANTINE_DIR / path.name
        shutil.move(str(path), str(dest))
        print("Quarantined:", path.name, "|", reason)

    bad_filenames = {path.name for path, _ in bad_images}
    dfd_manifest = dfd_manifest[~dfd_manifest["output_filename"].isin(bad_filenames)].reset_index(drop=True)
    dfd_manifest.to_csv(MANIFEST_PATH, index=False)
    print("Manifest updated after quarantine:", len(dfd_manifest), "rows remain.")

final_image_count = len(list(EXTERNAL_DFD_FAKE_DIR.glob("*.jpg")))
print("\nFinal evaluation images:", final_image_count)
assert final_image_count == len(dfd_manifest), "Manifest row count must match saved image count."


Corrupted/unreadable images found: 0

Final evaluation images: 8446


## 16. External DFD dataset / DataLoader

Uses the **same** normalization and evaluation transform as Lab 15/16 — resize to
224x224, ImageNet mean/std normalization. No random horizontal flip, no rotation, no
training augmentation of any kind: this is evaluation only. Reuses `FaceDataset`
unchanged; since `external_dfd/` has no `real/` subfolder, every yielded label is `1`
(FAKE), which Section 13 already confirmed is the ground truth for 100% of DFD.

In [21]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

eval_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

BATCH_SIZE = 16
NUM_WORKERS = 2  # configurable: safe default for Colab

dfd_dataset = FaceDataset(str(EXTERNAL_DFD_ROOT), transform=eval_transform)

dfd_loader = DataLoader(
    dfd_dataset, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available(),
)

print("DFD dataset size:", len(dfd_dataset))
print("DFD loader batches:", len(dfd_loader))
print("Unique labels present:", sorted(set(dfd_dataset.labels)), "(expect only [1] = FAKE)")

assert len(dfd_dataset) == len(dfd_manifest), "FaceDataset count must match the manifest."
assert set(dfd_dataset.labels) <= {1}, "DFD must contain FAKE (1) labels only - found a REAL (0) sample."

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

DFD dataset size: 8446
DFD loader batches: 528
Unique labels present: [1] (expect only [1] = FAKE)
Using device: cuda


## 17. Model loading

Checkpoints are **not** committed to GitHub (`.pth`/`.pt` are gitignored, and Lab 14-16's
`outputs/models/` only ships `.gitkeep`). Before running this section, upload each
checkpoint you want evaluated into this Colab runtime — either drag-and-drop into the
Colab file browser, or run the upload cell below — so the file exists at the exact path
each `..._CHECKPOINT` variable expects. No Google Drive is used or required.

This notebook evaluates whichever checkpoints are actually present; a missing checkpoint
prints a clear instruction and is skipped rather than crashing the whole run.

**Final notebook note:** This notebook is evaluation-only. It never retrains or fine-tunes the models. The `.pth` checkpoints must be supplied separately in the Colab runtime.


In [34]:
MODELS_DIR = PROJECT_ROOT / "outputs" / "models"
MODELS_DIR.mkdir(parents=True, exist_ok=True)

# Configure these paths to match wherever you upload/download each checkpoint.
EFFICIENTNET_BASELINE_CHECKPOINT = MODELS_DIR / "efficientnetb0_baseline_best.pth"      # Lab 15
EFFICIENTNET_BALANCED_CHECKPOINT = MODELS_DIR / "efficientnetb0_classbalanced_best.pth"  # Lab 16
MOBILENETV2_CHECKPOINT = MODELS_DIR / "mobilenetv2_baseline_best.pth"                    # Lab 14 (optional reference)

for name, path in [
    ("EfficientNet-B0 baseline (Lab 15)", EFFICIENTNET_BASELINE_CHECKPOINT),
    ("EfficientNet-B0 class-balanced (Lab 16)", EFFICIENTNET_BALANCED_CHECKPOINT),
    ("MobileNetV2 baseline (Lab 14, optional)", MOBILENETV2_CHECKPOINT),
]:
    print(f"{name}: {path} (exists={path.exists()})")

EfficientNet-B0 baseline (Lab 15): /content/Deepfake-Detection-KYC/outputs/models/efficientnetb0_baseline_best.pth (exists=True)
EfficientNet-B0 class-balanced (Lab 16): /content/Deepfake-Detection-KYC/outputs/models/efficientnetb0_classbalanced_best.pth (exists=True)
MobileNetV2 baseline (Lab 14, optional): /content/Deepfake-Detection-KYC/outputs/models/mobilenetv2_baseline_best.pth (exists=False)


In [35]:
# Optional: run this cell to upload a checkpoint from your local machine into
# this Colab runtime. Does not use Google Drive.
def upload_checkpoint(target_path: Path):
    if not IN_COLAB:
        print("Not running in Colab - place the checkpoint file manually at:", target_path)
        return
    from google.colab import files
    print(f"Select the .pth file to upload as: {target_path.name}")
    uploaded = files.upload()
    if not uploaded:
        print("No file uploaded.")
        return
    uploaded_name = next(iter(uploaded))
    target_path.parent.mkdir(parents=True, exist_ok=True)
    shutil.move(uploaded_name, target_path)
    print("Saved uploaded checkpoint to:", target_path)

# Example (uncomment and run the ones you need):
# upload_checkpoint(EFFICIENTNET_BASELINE_CHECKPOINT)
# upload_checkpoint(EFFICIENTNET_BALANCED_CHECKPOINT)
# upload_checkpoint(MOBILENETV2_CHECKPOINT)

In [36]:
def build_efficientnet_b0():
    model = efficientnet_b0(weights=None)
    num_features = model.classifier[1].in_features
    model.classifier[1] = nn.Linear(num_features, 2)
    return model


def build_mobilenet_v2():
    model = mobilenet_v2(weights=None)
    num_features = model.classifier[1].in_features
    model.classifier[1] = nn.Linear(num_features, 2)
    return model


MODEL_CONFIGS = {
    "EfficientNet-B0 baseline (Lab 15)": (build_efficientnet_b0, EFFICIENTNET_BASELINE_CHECKPOINT),
    "EfficientNet-B0 class-balanced (Lab 16)": (build_efficientnet_b0, EFFICIENTNET_BALANCED_CHECKPOINT),
    "MobileNetV2 baseline (Lab 14, optional)": (build_mobilenet_v2, MOBILENETV2_CHECKPOINT),
}

models_to_evaluate = {}

for name, (builder_fn, checkpoint_path) in MODEL_CONFIGS.items():
    if checkpoint_path.exists():
        models_to_evaluate[name] = (builder_fn, checkpoint_path)
        print(f"[available] {name} -> {checkpoint_path}")
    else:
        print(f"[MISSING]   {name} -> {checkpoint_path}")
        print(f"            Upload this checkpoint before evaluation (see upload_checkpoint() above).")

if not models_to_evaluate:
    print("\nNo checkpoints found yet. Upload at least the Lab 15 and Lab 16 checkpoints, then re-run this cell.")

[available] EfficientNet-B0 baseline (Lab 15) -> /content/Deepfake-Detection-KYC/outputs/models/efficientnetb0_baseline_best.pth
[available] EfficientNet-B0 class-balanced (Lab 16) -> /content/Deepfake-Detection-KYC/outputs/models/efficientnetb0_classbalanced_best.pth
[MISSING]   MobileNetV2 baseline (Lab 14, optional) -> /content/Deepfake-Detection-KYC/outputs/models/mobilenetv2_baseline_best.pth
            Upload this checkpoint before evaluation (see upload_checkpoint() above).


## 18. Model architecture verification

For every available checkpoint: instantiate the matching architecture, load the state
dict, run a forward-pass sanity check confirming the output dimension is 2, then set
`model.eval()`. No optimizer is created anywhere in this notebook, no `loss.backward()`
is called, and no training loop exists — this section only prepares models for
inference.

In [37]:
loaded_models = {}

if len(dfd_dataset) == 0:
    print("DFD dataset is empty - cannot run the forward-pass sanity check. "
          "Check Section 15's processing report before proceeding.")
else:
    sanity_images, _ = next(iter(dfd_loader))
    sanity_images = sanity_images.to(device)

    for name, (builder_fn, checkpoint_path) in models_to_evaluate.items():
        model = builder_fn()
        state_dict = torch.load(checkpoint_path, map_location=device)
        model.load_state_dict(state_dict)
        model = model.to(device)
        model.eval()

        with torch.no_grad():
            sanity_outputs = model(sanity_images)

        assert sanity_outputs.shape[1] == 2, (
            f"{name}: expected output dimension 2, got {sanity_outputs.shape[1]}."
        )

        loaded_models[name] = model
        print(f"Loaded and verified: {name} (output shape {tuple(sanity_outputs.shape)})")

print(f"\n{len(loaded_models)} model(s) ready for DFD evaluation.")

Loaded and verified: EfficientNet-B0 baseline (Lab 15) (output shape (16, 2))
Loaded and verified: EfficientNet-B0 class-balanced (Lab 16) (output shape (16, 2))

2 model(s) ready for DFD evaluation.


## 19. DFD evaluation (image level)

DFD contains **only FAKE examples** (verified in Section 13). Reporting standard
two-class metrics here would be misleading:

- **Accuracy and FAKE recall are numerically identical** on an all-FAKE set — both are
  just "fraction predicted FAKE" — so only FAKE recall / sensitivity is reported, not
  "accuracy" framed as if it reflected two-class discrimination.
- **FAKE precision is trivially 1.0** on an all-FAKE set (every true label is FAKE, so
  there can be no false positives by construction) and is reported only for completeness
  with an explicit note that it carries no information about model quality here.
- **ROC-AUC is undefined** with a single class present in the ground truth — computing it
  requires both classes. This notebook does not fabricate a value; it demonstrates the
  failure and explains why.

In [46]:
image_level_results = {}

for name, model in loaded_models.items():
    all_true = []
    all_pred = []
    all_prob_fake = []
    all_filenames = []

    with torch.no_grad():
        offset = 0
        for images, labels in dfd_loader:
            images = images.to(device, non_blocking=True)
            outputs = model(images)
            probabilities = torch.softmax(outputs, dim=1)
            predictions = outputs.argmax(dim=1)

            batch_size_actual = labels.size(0)
            batch_filenames = [
                Path(p).name for p in dfd_dataset.image_paths[offset: offset + batch_size_actual]
            ]
            offset += batch_size_actual

            all_true.extend(labels.numpy().tolist())
            all_pred.extend(predictions.cpu().numpy().tolist())
            all_prob_fake.extend(probabilities[:, 1].cpu().numpy().tolist())
            all_filenames.extend(batch_filenames)

    assert set(all_true) <= {1}, f"{name}: DFD ground truth must be all-FAKE (1)."

    total = len(all_true)
    correct = int(sum(p == 1 for p in all_pred))
    incorrect = total - correct
    fake_recall = recall_score(all_true, all_pred, pos_label=1, zero_division=0)
    fake_precision = precision_score(all_true, all_pred, pos_label=1, zero_division=0)

    cm = confusion_matrix(all_true, all_pred, labels=[0, 1])

    # Determine undefinedness explicitly from the class count rather than relying on
    # exception behavior: depending on the sklearn version, roc_auc_score on a
    # single-class y_true either raises ValueError OR returns nan with an
    # UndefinedMetricWarning (confirmed via a local synthetic check: sklearn 1.9.0
    # returns nan + warns, it does not raise). Checking len(set(all_true)) first
    # means we never depend on catching the "right" exception type and never let a
    # nan silently pass through as if it were a real score.
    if len(set(all_true)) < 2:
        roc_auc = None
        roc_auc_note = (
            "ROC-AUC is undefined: DFD ground truth contains only one class (FAKE), "
            "so a binary ROC curve cannot be computed. Not calculated (no value fabricated)."
        )
    else:
        try:
            roc_auc = roc_auc_score(all_true, all_prob_fake)
            roc_auc_note = None
        except ValueError as e:
            roc_auc = None
            roc_auc_note = (
                "ROC-AUC is undefined: DFD ground truth contains only one class (FAKE), "
                f"so a binary ROC curve cannot be computed. (sklearn error: {e})"
            )

    image_level_results[name] = {
        "total_images": total,
        "correct": correct,
        "incorrect": incorrect,
        "fake_recall": fake_recall,
        "fake_precision_trivial": fake_precision,
        "confusion_matrix": cm.tolist(),
        "roc_auc": roc_auc,
        "roc_auc_note": roc_auc_note,
        "prediction_counts": {"predicted_real": int(sum(p == 0 for p in all_pred)),
                               "predicted_fake": int(sum(p == 1 for p in all_pred))},
        "per_image": dict(zip(all_filenames, zip(all_pred, all_prob_fake))),
    }

    print("===================================")
    print(name)
    print("===================================")
    print(f"Total DFD images evaluated : {total}")
    print(f"Correct (predicted FAKE)   : {correct}")
    print(f"Incorrect (predicted REAL) : {incorrect}")
    print(f"FAKE recall / sensitivity  : {fake_recall*100:.2f}%")
    print(f"FAKE precision (trivial, uninformative on an all-FAKE set): {fake_precision*100:.2f}%")
    print(f"Confusion matrix [rows=true(REAL,FAKE), cols=pred(REAL,FAKE)]:\n{cm}")
    print(f"ROC-AUC: {roc_auc if roc_auc is not None else 'UNDEFINED'}")
    if roc_auc_note:
        print(f"  -> {roc_auc_note}")
    print()

EfficientNet-B0 baseline (Lab 15)
Total DFD images evaluated : 8446
Correct (predicted FAKE)   : 7700
Incorrect (predicted REAL) : 746
FAKE recall / sensitivity  : 91.17%
FAKE precision (trivial, uninformative on an all-FAKE set): 100.00%
Confusion matrix [rows=true(REAL,FAKE), cols=pred(REAL,FAKE)]:
[[   0    0]
 [ 746 7700]]
ROC-AUC: UNDEFINED
  -> ROC-AUC is undefined: DFD ground truth contains only one class (FAKE), so a binary ROC curve cannot be computed. Not calculated (no value fabricated).

EfficientNet-B0 class-balanced (Lab 16)
Total DFD images evaluated : 8446
Correct (predicted FAKE)   : 7294
Incorrect (predicted REAL) : 1152
FAKE recall / sensitivity  : 86.36%
FAKE precision (trivial, uninformative on an all-FAKE set): 100.00%
Confusion matrix [rows=true(REAL,FAKE), cols=pred(REAL,FAKE)]:
[[   0    0]
 [1152 7294]]
ROC-AUC: UNDEFINED
  -> ROC-AUC is undefined: DFD ground truth contains only one class (FAKE), so a binary ROC curve cannot be computed. Not calculated (no val

## 20. Video-level evaluation

**Aggregation rule:** for each DFD video, take the **mean FAKE probability** across every
evaluated face image belonging to that video (joined via `dfd_manifest.csv`'s
`video_file_path` -> `output_filename` mapping, which is exactly the video -> frame ->
face -> image trail recorded in Section 15). A video is predicted **FAKE** if that mean
probability is >= 0.5, else **REAL**. Since every DFD video is truly FAKE, "correct"
means the video-level prediction is FAKE.

Videos with **zero usable face images** (see Section 15's processing report — missing,
unreadable, no frames read, or no faces detected) cannot be scored and are reported
separately as **video-level evaluation failures**, never silently counted as REAL or
dropped from the denominator without disclosure.

In [47]:
video_level_results = {}

ok_videos = dfd_processing_log[dfd_processing_log["status"] == "ok"]["file_path"].tolist()
failed_videos = dfd_processing_log[dfd_processing_log["status"] != "ok"]

for name, result in image_level_results.items():
    per_image = result["per_image"]

    filename_to_video = dict(zip(dfd_manifest["output_filename"], dfd_manifest["video_file_path"]))

    video_probs = defaultdict(list)
    for filename, (pred, prob_fake) in per_image.items():
        video_path = filename_to_video.get(filename)
        if video_path is not None:
            video_probs[video_path].append(prob_fake)

    video_rows = []
    for video_path, probs in video_probs.items():
        mean_prob = float(np.mean(probs))
        video_pred = 1 if mean_prob >= 0.5 else 0
        video_rows.append({
            "video_file_path": video_path,
            "num_faces": len(probs),
            "mean_fake_probability": mean_prob,
            "video_prediction": "FAKE" if video_pred == 1 else "REAL",
            "correct": video_pred == 1,
        })

    video_df = pd.DataFrame(video_rows)
    evaluated_videos = len(video_df)
    correct_videos = int(video_df["correct"].sum()) if evaluated_videos else 0
    incorrect_videos = evaluated_videos - correct_videos
    fake_detection_rate = correct_videos / evaluated_videos if evaluated_videos else 0.0

    video_level_results[name] = {
        "total_dfd_videos": len(dfd_df),
        "evaluated_videos": evaluated_videos,
        "failed_videos": len(failed_videos),
        "correct_videos": correct_videos,
        "incorrect_videos": incorrect_videos,
        "fake_detection_rate": fake_detection_rate,
        "video_predictions": video_df.to_dict(orient="records"),
    }

    print("===================================")
    print(name, "- VIDEO LEVEL")
    print("===================================")
    print(f"Total DFD videos in split : {len(dfd_df)}")
    print(f"Evaluated (>=1 face)      : {evaluated_videos}")
    print(f"Evaluation failures       : {len(failed_videos)} (see dfd_processing_log for reasons)")
    print(f"Correct (predicted FAKE)  : {correct_videos}")
    print(f"Incorrect (predicted REAL): {incorrect_videos}")
    print(f"FAKE detection rate       : {fake_detection_rate*100:.2f}%")
    print()

if len(failed_videos) > 0:
    print(f"{len(failed_videos)} video(s) could not be evaluated at the video level:")
    print(failed_videos[["file_path", "status"]].to_string(index=False))

EfficientNet-B0 baseline (Lab 15) - VIDEO LEVEL
Total DFD videos in split : 1000
Evaluated (>=1 face)      : 994
Evaluation failures       : 6 (see dfd_processing_log for reasons)
Correct (predicted FAKE)  : 927
Incorrect (predicted REAL): 67
FAKE detection rate       : 93.26%

EfficientNet-B0 class-balanced (Lab 16) - VIDEO LEVEL
Total DFD videos in split : 1000
Evaluated (>=1 face)      : 994
Evaluation failures       : 6 (see dfd_processing_log for reasons)
Correct (predicted FAKE)  : 906
Incorrect (predicted REAL): 88
FAKE detection rate       : 91.15%

6 video(s) could not be evaluated at the video level:
                                                 file_path            status
DeepFakeDetection/01_11__secret_conversation__4OJNJLOO.mp4 no_faces_detected
DeepFakeDetection/02_01__secret_conversation__YVGY8LOK.mp4 no_faces_detected
DeepFakeDetection/02_07__secret_conversation__0IYV5DQ5.mp4 no_faces_detected
DeepFakeDetection/02_13__secret_conversation__CP5HFV3K.mp4 no_faces_detect

## 21. Compare Lab 15 vs Lab 16 (and MobileNetV2, if evaluated)

In [48]:
print(f"{'Model':<42}{'DFD videos':>12}{'Correct':>10}{'Incorrect':>11}{'FAKE det. rate':>16}")
for name, r in video_level_results.items():
    print(
        f"{name:<42}"
        f"{r['evaluated_videos']:>12d}"
        f"{r['correct_videos']:>10d}"
        f"{r['incorrect_videos']:>11d}"
        f"{r['fake_detection_rate']*100:>15.2f}%"
    )

if "EfficientNet-B0 baseline (Lab 15)" in video_level_results and "EfficientNet-B0 class-balanced (Lab 16)" in video_level_results:
    lab15_rate = video_level_results["EfficientNet-B0 baseline (Lab 15)"]["fake_detection_rate"]
    lab16_rate = video_level_results["EfficientNet-B0 class-balanced (Lab 16)"]["fake_detection_rate"]
    delta = lab16_rate - lab15_rate
    print(f"\nLab 16 vs Lab 15 DFD FAKE detection rate delta: {delta*100:+.2f} pp")

Model                                       DFD videos   Correct  Incorrect  FAKE det. rate
EfficientNet-B0 baseline (Lab 15)                  994       927         67          93.26%
EfficientNet-B0 class-balanced (Lab 16)            994       906         88          91.15%

Lab 16 vs Lab 15 DFD FAKE detection rate delta: -2.11 pp


## 22. Core test vs. external DFD

**Core test accuracy is a two-class metric** (core test contains both REAL and FAKE).
**DFD is all-FAKE**, so its only meaningful external metric is FAKE detection rate /
recall — comparing DFD's detection rate against core-test **accuracy** would compare two
different things. The valid, compatible comparison is:

**core-test FAKE recall (a one-class-conditional metric) vs. DFD FAKE detection rate
(the same kind of quantity, on a different, external population of FAKE videos).**

Core-test accuracy and ROC-AUC are still shown below for reference, but are explicitly
*not* compared directly against DFD's FAKE-only metric.

In [49]:
core_test_reference = {
    "EfficientNet-B0 baseline (Lab 15)": {
        "test_accuracy": 0.9041, "test_recall_fake": 0.9717, "test_roc_auc": 0.9055,
        "source": "hardcoded from completed Lab 15 results",
    },
    "MobileNetV2 baseline (Lab 14, optional)": {
        "test_accuracy": 0.9024, "test_recall_fake": 0.9775, "test_roc_auc": 0.9020,
        "source": "hardcoded from completed Lab 14 results",
    },
}

lab16_results_path = PROJECT_ROOT / "outputs" / "results" / "lab16_classbalanced_efficientnet_results.json"
if lab16_results_path.exists():
    with open(lab16_results_path) as f:
        lab16_results = json.load(f)
    core_test_reference["EfficientNet-B0 class-balanced (Lab 16)"] = {
        "test_accuracy": lab16_results.get("test_accuracy"),
        "test_recall_fake": lab16_results.get("test_recall_fake"),
        "test_roc_auc": lab16_results.get("test_roc_auc"),
        "source": f"loaded from {lab16_results_path}",
    }
    print("Loaded Lab 16 core-test results from this runtime:", lab16_results_path)
else:
    print(f"Lab 16 results file not found at {lab16_results_path} in this runtime.")
    print("Core-test comparison for Lab 16 will be shown as unavailable; "
          "upload/generate that file to enable it.")

print()
print(f"{'Model':<42}{'Core Acc':>10}{'Core FAKE R':>13}{'Core ROC-AUC':>14}{'DFD FAKE det.':>15}")
for name in video_level_results:
    core = core_test_reference.get(name)
    dfd_rate = video_level_results[name]["fake_detection_rate"] * 100
    if core and core.get("test_accuracy") is not None:
        print(
            f"{name:<42}"
            f"{core['test_accuracy']*100:>9.2f}%"
            f"{core['test_recall_fake']*100:>12.2f}%"
            f"{core['test_roc_auc']:>14.4f}"
            f"{dfd_rate:>14.2f}%"
        )
    else:
        print(f"{name:<42}{'N/A':>10}{'N/A':>13}{'N/A':>14}{dfd_rate:>14.2f}%")

Lab 16 results file not found at /content/Deepfake-Detection-KYC/outputs/results/lab16_classbalanced_efficientnet_results.json in this runtime.
Core-test comparison for Lab 16 will be shown as unavailable; upload/generate that file to enable it.

Model                                       Core Acc  Core FAKE R  Core ROC-AUC  DFD FAKE det.
EfficientNet-B0 baseline (Lab 15)             90.41%       97.17%        0.9055         93.26%
EfficientNet-B0 class-balanced (Lab 16)          N/A          N/A           N/A         91.15%


## 23. Save results

In [50]:
import csv

RESULTS_DIR = PROJECT_ROOT / "outputs" / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

timestamp = datetime.now(timezone.utc).isoformat()

full_results = {
    "timestamp_utc": timestamp,
    "seed": SEED,
    "dataset": "DeepFakeDetection (DFD), external held-out",
    "split_file": str(DFD_SPLIT_CSV),
    "total_dfd_videos": len(dfd_df),
    "preprocessing_config": {
        "frame_interval": FRAME_INTERVAL,
        "max_frames_per_video": MAX_FRAMES_PER_VIDEO,
        "image_size": 224,
        "transforms": "resize(224x224) + ImageNet normalize, no augmentation",
        "batch_size": BATCH_SIZE,
        "num_workers": NUM_WORKERS,
    },
    "aggregation_method": "video-level prediction = mean FAKE probability across all evaluated "
                           "face images for that video, thresholded at 0.5",
    "processing_summary": {
        "status_counts": dfd_processing_log["status"].value_counts().to_dict(),
        "total_frames_sampled": int(dfd_processing_log["frames_read"].sum()),
        "total_faces_saved": int(dfd_processing_log["faces_saved"].sum()),
        "final_evaluation_images": len(dfd_manifest),
    },
    "models": {},
}

csv_rows = []

for name in loaded_models:
    checkpoint_path = models_to_evaluate[name][1]
    img_result = image_level_results[name]
    vid_result = video_level_results[name]

    full_results["models"][name] = {
        "checkpoint_path": str(checkpoint_path),
        "image_level": {k: v for k, v in img_result.items() if k != "per_image"},
        "video_level": vid_result,
        "core_test_reference": core_test_reference.get(name),
    }

    csv_rows.append({
        "model": name,
        "dataset": "DeepFakeDetection (DFD)",
        "num_videos": len(dfd_df),
        "num_evaluated_videos": vid_result["evaluated_videos"],
        "num_failed_videos": vid_result["failed_videos"],
        "num_evaluated_images": img_result["total_images"],
        "correct_videos": vid_result["correct_videos"],
        "incorrect_videos": vid_result["incorrect_videos"],
        "fake_detection_rate": vid_result["fake_detection_rate"],
        "image_level_fake_recall": img_result["fake_recall"],
        "predicted_real_images": img_result["prediction_counts"]["predicted_real"],
        "predicted_fake_images": img_result["prediction_counts"]["predicted_fake"],
        "checkpoint_path": str(checkpoint_path),
        "aggregation_method": "mean_fake_probability_threshold_0.5",
        "timestamp_utc": timestamp,
    })

results_json_path = RESULTS_DIR / "lab17_dfd_external_evaluation.json"
with open(results_json_path, "w") as f:
    json.dump(full_results, f, indent=2)

results_csv_path = RESULTS_DIR / "lab17_dfd_external_evaluation.csv"
if csv_rows:
    with open(results_csv_path, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=list(csv_rows[0].keys()))
        writer.writeheader()
        writer.writerows(csv_rows)
else:
    with open(results_csv_path, "w") as f:
        f.write("model,dataset,num_videos,num_evaluated_videos,num_failed_videos,num_evaluated_images,"
                 "correct_videos,incorrect_videos,fake_detection_rate,image_level_fake_recall,"
                 "predicted_real_images,predicted_fake_images,checkpoint_path,aggregation_method,timestamp_utc\n")

print("Saved results JSON to:", results_json_path)
print("Saved results CSV to: ", results_csv_path)

Saved results JSON to: /content/Deepfake-Detection-KYC/outputs/results/lab17_dfd_external_evaluation.json
Saved results CSV to:  /content/Deepfake-Detection-KYC/outputs/results/lab17_dfd_external_evaluation.csv


## 24. Research interpretation

Auto-generated from the computed numbers above. This section deliberately avoids
claiming "the model generalizes well" from a bare percentage — it compares DFD's FAKE
detection rate against each model's **own core-test FAKE recall** to look for evidence of
distribution shift, and states the DFD-only limitation explicitly every time.

In [51]:
print("===================================")
print("RESEARCH INTERPRETATION")
print("===================================")

STRONG_THRESHOLD = 0.90
WEAK_THRESHOLD = 0.75

for name, vid_result in video_level_results.items():
    dfd_rate = vid_result["fake_detection_rate"]
    core = core_test_reference.get(name)

    print(f"\n{name}:")

    if dfd_rate >= STRONG_THRESHOLD:
        strength = "strong"
    elif dfd_rate >= WEAK_THRESHOLD:
        strength = "moderate"
    else:
        strength = "weak"
    print(f"  External FAKE detection on DFD is {strength} ({dfd_rate*100:.2f}% of {vid_result['evaluated_videos']} evaluated videos).")

    if core and core.get("test_recall_fake") is not None:
        shift = core["test_recall_fake"] - dfd_rate
        print(f"  Core-test FAKE recall: {core['test_recall_fake']*100:.2f}% | DFD FAKE detection: {dfd_rate*100:.2f}%")
        if abs(shift) < 0.03:
            print(f"  Difference is small ({shift*100:+.2f} pp) - limited evidence of distribution shift on the FAKE class.")
        elif shift > 0:
            print(f"  DFD detection is {shift*100:.2f} pp LOWER than core-test FAKE recall - evidence the model is "
                  "somewhat less sensitive to DFD's manipulation style/identities than to the core split's.")
        else:
            print(f"  DFD detection is {-shift*100:.2f} pp HIGHER than core-test FAKE recall.")
    else:
        print("  Core-test FAKE recall unavailable in this runtime - cannot compare against internal performance.")

    print(f"  Evaluation failures: {vid_result['failed_videos']} video(s) had no usable detected face and were "
          "excluded from this rate (reported separately, not treated as REAL).")
    print("  LIMITATION: DFD contains only FAKE videos. This experiment cannot measure REAL-vs-FAKE "
          "discrimination, false-positive behavior, or ROC-AUC by itself - it only measures sensitivity "
          "to this specific external FAKE distribution.")

if "EfficientNet-B0 baseline (Lab 15)" in video_level_results and "EfficientNet-B0 class-balanced (Lab 16)" in video_level_results:
    lab15_rate = video_level_results["EfficientNet-B0 baseline (Lab 15)"]["fake_detection_rate"]
    lab16_rate = video_level_results["EfficientNet-B0 class-balanced (Lab 16)"]["fake_detection_rate"]
    print("\nClass balancing (Lab 16) vs. baseline (Lab 15) on external DFD:")
    if lab16_rate > lab15_rate:
        print(f"  Class-balanced training INCREASED external FAKE detection by {(lab16_rate-lab15_rate)*100:.2f} pp.")
    elif lab16_rate < lab15_rate:
        print(f"  Class-balanced training REDUCED external FAKE detection by {(lab15_rate-lab16_rate)*100:.2f} pp.")
    else:
        print("  Class-balanced training left external FAKE detection unchanged.")
    print("  Note: since Lab 16 down-weights FAKE relative to REAL during training to fix the core 1:5 "
          "imbalance, some reduction in raw FAKE sensitivity here would be an expected, not necessarily "
          "bad, side effect - it should be read together with Lab 16's own core-test REAL/FAKE trade-off.")

RESEARCH INTERPRETATION

EfficientNet-B0 baseline (Lab 15):
  External FAKE detection on DFD is strong (93.26% of 994 evaluated videos).
  Core-test FAKE recall: 97.17% | DFD FAKE detection: 93.26%
  DFD detection is 3.91 pp LOWER than core-test FAKE recall - evidence the model is somewhat less sensitive to DFD's manipulation style/identities than to the core split's.
  Evaluation failures: 6 video(s) had no usable detected face and were excluded from this rate (reported separately, not treated as REAL).
  LIMITATION: DFD contains only FAKE videos. This experiment cannot measure REAL-vs-FAKE discrimination, false-positive behavior, or ROC-AUC by itself - it only measures sensitivity to this specific external FAKE distribution.

EfficientNet-B0 class-balanced (Lab 16):
  External FAKE detection on DFD is strong (91.15% of 994 evaluated videos).
  Core-test FAKE recall unavailable in this runtime - cannot compare against internal performance.
  Evaluation failures: 6 video(s) had no usab

## 25. Reproducibility / configuration summary

In [52]:
print("===================================")
print("LAB 17 CONFIGURATION SUMMARY")
print("===================================")
print("Seed                   :", SEED)
print("Image size              : 224x224")
print("Preprocessing            : Haar-cascade face detection + resize(224x224), "
      "frame_interval=", FRAME_INTERVAL, ", max_frames_per_video=", MAX_FRAMES_PER_VIDEO)
print("Evaluation transform    : resize(224x224) + ImageNet normalize, no augmentation")
print("Aggregation method       : mean FAKE probability per video, threshold 0.5")
print("Dataset                  : DeepFakeDetection (DFD), external, all-FAKE")
print("Total DFD videos         :", len(dfd_df))
print("Final evaluation images  :", len(dfd_manifest))
print()
print("Models evaluated:")
for name in loaded_models:
    print(f"  - {name}")
    print(f"      checkpoint: {models_to_evaluate[name][1]}")

LAB 17 CONFIGURATION SUMMARY
Seed                   : 42
Image size              : 224x224
Preprocessing            : Haar-cascade face detection + resize(224x224), frame_interval= 10 , max_frames_per_video= 5
Evaluation transform    : resize(224x224) + ImageNet normalize, no augmentation
Aggregation method       : mean FAKE probability per video, threshold 0.5
Dataset                  : DeepFakeDetection (DFD), external, all-FAKE
Total DFD videos         : 1000
Final evaluation images  : 8446

Models evaluated:
  - EfficientNet-B0 baseline (Lab 15)
      checkpoint: /content/Deepfake-Detection-KYC/outputs/models/efficientnetb0_baseline_best.pth
  - EfficientNet-B0 class-balanced (Lab 16)
      checkpoint: /content/Deepfake-Detection-KYC/outputs/models/efficientnetb0_classbalanced_best.pth


## 26. Final summary

In [53]:
print("===================================")
print("LAB 17 FINAL SUMMARY - EXTERNAL DFD EVALUATION")
print("===================================")
print(f"{'Model':<42}{'DFD FAKE det.':>15}{'Img FAKE recall':>17}{'Videos eval/fail':>18}")
for name in video_level_results:
    vid = video_level_results[name]
    img = image_level_results[name]
    print(
        f"{name:<42}"
        f"{vid['fake_detection_rate']*100:>14.2f}%"
        f"{img['fake_recall']*100:>16.2f}%"
        f"{vid['evaluated_videos']:>10d}/{vid['failed_videos']:<6d}"
    )

print()
print("Results saved to:")
print(" -", results_json_path)
print(" -", results_csv_path)
print()
print("Next step: final model selection across Lab 15/16 using both core-test and DFD "
      "external evidence, followed by final analysis, report, presentation, and README.")

LAB 17 FINAL SUMMARY - EXTERNAL DFD EVALUATION
Model                                       DFD FAKE det.  Img FAKE recall  Videos eval/fail
EfficientNet-B0 baseline (Lab 15)                  93.26%           91.17%       994/6     
EfficientNet-B0 class-balanced (Lab 16)            91.15%           86.36%       994/6     

Results saved to:
 - /content/Deepfake-Detection-KYC/outputs/results/lab17_dfd_external_evaluation.json
 - /content/Deepfake-Detection-KYC/outputs/results/lab17_dfd_external_evaluation.csv

Next step: final model selection across Lab 15/16 using both core-test and DFD external evidence, followed by final analysis, report, presentation, and README.
